In [1]:
import base64, json
from enum import Enum
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, ConfigDict, Field
from tqdm.auto import tqdm

/home/rakesh/Downloads/ShortTok/3.ObjectDetection/ObjectDetection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
def labels(name, values):
    return Enum(name, {value: value for value in values})

def detection_model(vocab):
    class ObjectDetectionOutput(BaseModel):
        model_config = ConfigDict(extra="forbid")
        relation: labels("relation", vocab["relation"])
        foreground: labels("foreground", vocab["foreground"])
        background: labels("background", vocab["background"])
    return ObjectDetectionOutput

## Groq: Qwen3.8-27b

In [ ]:
client = Groq()

In [4]:
def predict(paths, vocab):
    Output = detection_model(vocab)

    class Batch(BaseModel):
        model_config = ConfigDict(extra="forbid")
        labels: list[Output] = Field(min_length=len(paths), max_length=len(paths))

    content = [{"type": "text", "text": "Label each image in order. Return one labels object per image."}]
    for path in paths:
        image = base64.b64encode(path.read_bytes()).decode()
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image}"}})
    reply = client.chat.completions.create(
        model="qwen/qwen3.8-27b",
        messages=[{"role": "user", "content": content}],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "labels", "strict": True, "schema": Batch.model_json_schema()},
        },
        reasoning_effort="none",
        temperature=0,
        max_completion_tokens=256,
    )
    return Batch.model_validate_json(reply.choices[0].message.content).model_dump(mode="json")["labels"]

In [ ]:
batch_size = 2
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    groups = [rows[i:i + 2] for i in range(0, len(rows), batch_size)]
    for group in tqdm(groups, desc=split):
        paths = [Path(f"data/{split}/images") / row["file"] for row in group]
        for row, pred in zip(group, predict_local(paths, vocab)):
            tqdm.write(f"{split} {row['file']} {pred}")

In [ ]:
batch_size = 3
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    groups = [rows[i:i + 3] for i in range(0, len(rows), batch_size)]
    for group in tqdm(groups, desc=split):
        paths = [Path(f"data/{split}/images") / row["file"] for row in group]
        for row, pred in zip(group, predict(paths, vocab)):
            tqdm.write(f"{split} {row['file']} {pred}")

ecommerce:   0%|          | 0/13 [00:00<?, ?it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.8-27b` in organization `org_01hy29aamzfrtrdgq7fjwfcg9y` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199154, Requested 6385. Please try again in 39m52.848s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Local GPU: Qwen3-VL-2B, 4-bit, batch of 2

In [4]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL = "Qwen/Qwen3-VL-2B-Instruct"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL,
    cache_dir="models",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL, cache_dir="models", max_pixels=401408)
processor.tokenizer.padding_side = "left"

Loading weights: 100%|██████████| 625/625 [00:01<00:00, 374.90it/s]


In [5]:
def pick(value, choices):
    text = str(value).strip()
    if text in choices:
        return text
    for part in text.replace(",", " ").split():
        if part in choices:
            return part
    return ""

def predict_local(paths, vocab):
    prompt = (
        'Reply with only {"foreground":"...","background":"...","relation":"..."}. One label each, no commas inside values.\n'
        f"foreground: {', '.join(vocab['foreground'])}\n"
        f"background: {', '.join(vocab['background'])}\n"
        f"relation: {', '.join(vocab['relation'])}"
    )
    messages = [[{
        "role": "user",
        "content": [
            {"type": "image", "image": str(path.resolve())},
            {"type": "text", "text": prompt},
        ],
    }] for path in paths]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", padding=True,
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=160, do_sample=False,
            stop_strings=["}"], tokenizer=processor.tokenizer,
        )
    texts = processor.batch_decode(
        [out[len(src):] for src, out in zip(inputs.input_ids, generated)],
        skip_special_tokens=True,
    )
    preds = []
    for text in texts:
        if text.count('"') % 2:
            text += '"'
        if not text.rstrip().endswith("}"):
            text += "}"
        start, end = text.find("{"), text.rfind("}")
        try:
            obj = json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            obj = {}
        preds.append({key: pick(obj.get(key, text), vocab[key]) for key in vocab})
    return preds

In [6]:
results = []
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    groups = [rows[i:i + 2] for i in range(0, len(rows), 2)]
    for group in tqdm(groups, desc=split):
        paths = [Path(f"data/{split}/images") / row["file"] for row in group]
        for row, pred in zip(group, predict_local(paths, vocab)):
            results.append((split, row, pred))
            tqdm.write(f"{split} {row['file']} {pred}")

ecommerce:   0%|          | 0/19 [00:00<?, ?it/s][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
/home/rakesh/Downloads/ShortTok/3.ObjectDetection/ObjectDetection/.venv/lib/python3.12/site-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
ecommerce:   5%|▌         | 1/19 [00:01<00:34,  1.92s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


ecommerce A_black_coffee_mug_on_an_office_desk_with_a_blurred_laptop_in_the_background.jpg {'foreground': 'coffee_mug', 'background': 'blurred_laptop', 'relation': 'object_with_background'}
ecommerce a_camera_is_attached_to_a_tripod_with_a_mountain_in_the_background.jpg {'foreground': 'camera', 'background': 'urban_architecture', 'relation': 'object_with_background'}


ecommerce:  11%|█         | 2/19 [00:03<00:26,  1.56s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


ecommerce a_camera_on_a_tripod_with_a_mountain_in_the_background.jpg {'foreground': 'camera', 'background': 'mountain', 'relation': 'object_with_background'}
ecommerce A_close_up_shot_of_a_fountain_pen_with_an_ink_bottle_highlighted_by_warm_bokeh_lights_in_the_background.jpg {'foreground': 'pen', 'background': 'blurred_lights', 'relation': 'object_with_background'}


ecommerce:  16%|█▌        | 3/19 [00:04<00:22,  1.43s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


...

news Joe_Biden_at_Buffalo.jpg {'foreground': 'Joe_Biden', 'background': 'White_House', 'relation': 'people_at_event'}
news Joe_Biden_at_Oval_Office.jpg {'foreground': 'Joe_Biden', 'background': 'White_House', 'relation': 'people_at_location'}


news:  88%|████████▊ | 7/8 [00:12<00:01,  1.71s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


news Joe_Biden_at_State_of_the_Union.jpg {'foreground': 'Joe_Biden', 'background': 'White_House', 'relation': 'people_at_event'}
news Joe_Biden_at_White_House_Correspondents_Dinner.jpg {'foreground': 'Joe_Biden', 'background': 'White_House', 'relation': 'people_at_location'}


news: 100%|██████████| 8/8 [00:13<00:00,  1.72s/it]

news Mr_Bezos_at_Florida.jpg {'foreground': 'Jeff_Bezos', 'background': '', 'relation': 'people_at_event'}


In [7]:
from sklearn.metrics import classification_report

def pick(value, choices):
    text = str(value).strip()
    if text in choices:
        return text
    for part in text.replace(",", " ").split():
        if part in choices:
            return part
    return ""

keys = ("foreground", "background", "relation")
for split in ("ecommerce", "news"):
    pairs = [(pred, row) for name, row, pred in results if name == split]
    labels = {key: sorted({row[key] for _, row in pairs}) for key in keys}
    pairs = [({key: pick(pred.get(key, ""), labels[key]) for key in keys}, row) for pred, row in pairs]
    print(split, "exact", round(sum(all(pred[k] == row[k] for k in keys) for pred, row in pairs) / len(pairs), 3))
    for key in keys:
        gold = [row[key] for _, row in pairs]
        pred = [item[key] for item, _ in pairs]
        print(key)
        print(classification_report(gold, pred, labels=labels[key], zero_division=0))

ecommerce exact 0.605
foreground
                precision    recall  f1-score   support

   alarm_clock       1.00      1.00      1.00         1
      backpack       1.00      1.00      1.00         1
          book       1.00      1.00      1.00         1
        camera       1.00      1.00      1.00         9
    coffee_mug       0.89      1.00      0.94         8
           cup       0.00      0.00      0.00         1
   flower_vase       1.00      1.00      1.00         1
       handbag       1.00      1.00      1.00         1
           jar       1.00      1.00      1.00         1
        kettle       1.00      1.00      1.00         1
          lamp       1.00      1.00      1.00         1
        laptop       1.00      1.00      1.00         1
           pen       1.00      1.00      1.00         1
perfume_bottle       1.00      1.00      1.00         1
         phone       1.00      1.00      1.00         2
          soap       1.00      1.00      1.00         1
    sunglasses